# Tutorial: `is_discrete`, `quartiles`, `deciles`, and `tabulate`

This notebook walks through four utility functions added to the calpoly-symbulate fork.

In [2]:
from symbulate import *

---
## `is_discrete`

Returns `True` if more than 80% of distinct values in a collection appear more than once. This repeat-count heuristic detects whether simulated results came from a discrete distribution (where outcomes repeat) vs. a continuous one (where nearly every value is unique). It is not tied to whether values are integers.

In [3]:
# Most distinct values repeat -> True
is_discrete([0, 1, 1, 0, 1])

True

In [4]:
# All values unique -> False, even if they are whole numbers
is_discrete([1.0, 2.0, 3.0])

False

In [5]:
# Continuous-looking values — all unique, so False
is_discrete([0.5, 1.2, 3.7])

False

In [ ]:
# Rare tail values with count=1 are fine — 9 of 10 distinct values repeat (90% > 80%)
# mirrors a Poisson-like sample where common outcomes repeat but a rare tail value appears once
is_discrete([0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9])

In [ ]:
# Heuristic is type-agnostic — non-numeric hashable values work the same way
is_discrete(["heads", "heads", "tails", "tails", "heads"])

In [6]:
# Works directly on simulated Results
bernoulli_sims = RV(Bernoulli(0.5)).sim(500)
normal_sims    = RV(Normal(0, 1)).sim(500)

print("Bernoulli:", is_discrete(bernoulli_sims))
print("Normal:   ", is_discrete(normal_sims))

Bernoulli: True
Normal:    False


In [7]:
# TypeError if you pass an RV instead of simulated results
try:
    is_discrete(RV(Poisson(3)))
except TypeError as e:
    print(e)

is_discrete requires simulated results, not an RV. Use X.sim(n) first.


---
## `quartiles`

Returns a dict of the five-number summary: min (0%), Q1 (25%), median (50%), Q3 (75%), and max (100%). These are cut points, not bins.

In [7]:
# Basic example
quartiles([1, 2, 3, 4, 5])

{0.0: 1.0, 0.25: 2.0, 0.5: 3.0, 0.75: 4.0, 1.0: 5.0}

In [8]:
# On simulated data from an Exponential distribution
exp_sims = RV(Exponential(1)).sim(1000)
quartiles(exp_sims)

{0.0: 0.0003148721181035176,
 0.25: 0.3061387897802781,
 0.5: 0.7440610327243558,
 0.75: 1.4005747069883345,
 1.0: 7.802553430296139}

In [9]:
# Access individual quartile values
q = quartiles(exp_sims)
print("Median:", q[0.50])
print("IQR:   ", q[0.75] - q[0.25])

Median: 0.7440610327243558
IQR:    1.0944359172080564


In [10]:
# ValueError on empty input
try:
    quartiles([])
except ValueError as e:
    print(e)

quartiles requires a non-empty collection.


---
## `deciles`

Like `quartiles` but returns 11 cut points at every 10th percentile (0%, 10%, 20%, ..., 100%). Useful for a finer-grained picture of the distribution.

In [11]:
# Basic example
deciles([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

{0.0: 1.0,
 0.1: 1.9,
 0.2: 2.8,
 0.3: 3.6999999999999997,
 0.4: 4.6,
 0.5: 5.5,
 0.6: 6.3999999999999995,
 0.7: 7.3,
 0.8: 8.2,
 0.9: 9.1,
 1.0: 10.0}

In [12]:
# On simulated Normal data
norm_sims = RV(Normal(0, 1)).sim(1000)
deciles(norm_sims)

{0.0: -3.3588111361926876,
 0.1: -1.2718201022408275,
 0.2: -0.8365817279095352,
 0.3: -0.5245476759380229,
 0.4: -0.25697222647823137,
 0.5: 0.06636425020360127,
 0.6: 0.3278090480551789,
 0.7: 0.5357750939016852,
 0.8: 0.814006515724337,
 0.9: 1.2441142872237907,
 1.0: 2.849247847873136}

In [13]:
# Access individual decile values
d = deciles(norm_sims)
print("10th percentile:", d[0.1])
print("90th percentile:", d[0.9])

10th percentile: -1.2718201022408275
90th percentile: 1.2441142872237907


In [14]:
# Compare quartiles vs deciles — deciles give more detail
exp_sims = RV(Exponential(1)).sim(1000)
print("Quartiles:", quartiles(exp_sims))
print()
print("Deciles:  ", deciles(exp_sims))

Quartiles: {0.0: 0.001238458923202941, 0.25: 0.2994677748001575, 0.5: 0.6761355858981704, 0.75: 1.3872919214995731, 1.0: 7.302345834877318}

Deciles:   {0.0: 0.001238458923202941, 0.1: 0.10191862246520915, 0.2: 0.24037953438867724, 0.3: 0.3559236838974271, 0.4: 0.5021263607101515, 0.5: 0.6761355858981704, 0.6: 0.8961083919220832, 0.7: 1.1672618191961512, 0.8: 1.593034495094413, 0.9: 2.189773103642638, 1.0: 7.302345834877318}


---
## `tabulate`

Counts how many times each outcome appears in simulated results. Available on both `Results` (probability space) and `RVResults` (random variable).

**Parameters:**
- `outcomes` — list of outcomes to include; missing ones get count 0. Ignored when `bin=True`.
- `normalize` — if `True`, return relative frequencies instead of counts.
- `bin` — if `True`, group continuous results into equal-width bins.
- `nbins` — number of bins (default 10). Only used when `bin=True`.
- `binwidth` — bin width; number of bins is inferred from the data range. Only used when `bin=True`.

`nbins` and `binwidth` cannot be used together.

In [15]:
# Basic counts — discrete outcomes
RV(Bernoulli(0.5)).sim(200).tabulate()

0,97
1,103
Total,200


In [16]:
# normalize=True gives relative frequencies
RV(Bernoulli(0.5)).sim(200).tabulate(normalize=True)

0,0.495
1,0.505
Total,1.0


In [17]:
# outcomes= ensures all values appear even if not simulated
RV(DiscreteUniform(1, 6)).sim(20).tabulate(outcomes=[1, 2, 3, 4, 5, 6])

1,2
2,1
3,2
4,5
5,6
6,4
Total,20


In [18]:
# bin=True groups continuous results into 10 equal-width bins by default
# Bins are [a, b) except the last which is [a, b]
RV(Normal(0, 1)).sim(1000).tabulate(bin=True)

"[-3.099, -2.462)",11
"[-2.462, -1.826)",32
"[-1.826, -1.189)",74
"[-1.189, -0.5522)",174
"[-0.5522, 0.08451)",245
"[0.08451, 0.7212)",218
"[0.7212, 1.358)",149
"[1.358, 1.995)",73
"[1.995, 2.631)",18
"[2.631, 3.268]",6
Total,1000


In [19]:
# nbins= sets a custom number of bins
RV(Normal(0, 1)).sim(1000).tabulate(bin=True, nbins=5)

"[-2.877, -1.64)",48
"[-1.64, -0.4021)",278
"[-0.4021, 0.8355)",461
"[0.8355, 2.073)",192
"[2.073, 3.311]",21
Total,1000


In [20]:
# binwidth= sets a fixed bin width; number of bins is inferred from the data range
RV(Normal(0, 1)).sim(1000).tabulate(bin=True, binwidth=0.5)

"[-3.037, -2.571)",2
"[-2.571, -2.106)",14
"[-2.106, -1.64)",31
"[-1.64, -1.175)",78
"[-1.175, -0.709)",95
"[-0.709, -0.2434)",181
"[-0.2434, 0.2222)",186
"[0.2222, 0.6878)",161
"[0.6878, 1.153)",115
"[1.153, 1.619)",81
"[1.619, 2.085)",30


In [21]:
# bin=True + normalize=True gives the proportion of observations in each bin
RV(Exponential(1)).sim(1000).tabulate(bin=True, normalize=True)

"[0.0003619, 0.828)",0.569
"[0.828, 1.656)",0.231
"[1.656, 2.483)",0.111
"[2.483, 3.311)",0.047
"[3.311, 4.139)",0.026
"[4.139, 4.966)",0.012
"[4.966, 5.794)",0.003
"[5.794, 6.622)",0.0
"[6.622, 7.449)",0.0
"[7.449, 8.277]",0.001
Total,1.0


In [22]:
# ValueError if both nbins and binwidth are specified
try:
    RV(Normal(0, 1)).sim(100).tabulate(bin=True, nbins=5, binwidth=0.5)
except ValueError as e:
    print(e)

Cannot specify both nbins and binwidth. Use nbins to set the number of bins, or binwidth to set the bin width.


In [23]:
# UserWarning if nbins or binwidth are passed without bin=True
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    RV(Normal(0, 1)).sim(100).tabulate(nbins=5)
    print(w[0].message)

nbins and binwidth have no effect when bin=False.
